# numpy за один присест

Всё, что нужно, чтобы читать и писать код курса. Порядок — от «что это
вообще» до конструкций, которые встретятся в `minipipe`.

| # | О чём |
|---|---|
| 1 | Массив — не список |
| 2 | Форма и оси |
| 3 | Векторизация вместо цикла |
| 4 | Broadcasting |
| 5 | Индексация: срезы, маски, списки индексов |
| 6 | View против copy |
| 7 | Полезные функции |
| 8 | Случайность |
| 9 | Где это в `minipipe` |

In [ ]:
import numpy as np

print("numpy", np.__version__)

## 1. Массив — не список

У списка элементы — разные объекты, разбросанные по памяти. У массива —
**один тип на всех**, подряд, одним куском.

In [ ]:
values = np.array([1, 2, 3])
print(values, values.dtype)

floats = np.array([1.0, 2.0, 3.0])
print(floats, floats.dtype)

# тип фиксирован: дробное в целочисленный массив не влезет
values[0] = 3.9
print("после записи 3.9 в int-массив:", values)

Отсюда два следствия: массив занимает меньше памяти и операции над ним
выполняются одним вызовом C-кода, а не циклом на Python.

Тип задаётся явно, когда это важно:

In [ ]:
print(np.array([1, 2, 3], dtype=np.float64))
print(np.zeros(3), np.ones(3, dtype=np.int64))
print(np.empty(3, dtype=np.int64))      # не инициализирован, там мусор
print(np.arange(5, dtype=float))

## 2. Форма и оси

`shape` — кортеж: сколько элементов вдоль каждой оси.
**`axis` — это индекс в этом кортеже.**

In [ ]:
table = np.arange(12, dtype=float).reshape(4, 3)
print(table)
print("shape:", table.shape, "| ndim:", table.ndim, "| всего:", table.size)

```
shape = (4, 3)
          │  └── axis=1
          └───── axis=0
```

Отсюда правило, которого хватает навсегда:

> **Свёртка по `axis=k` выбрасывает `k`-ю позицию из `shape`.**

`(4, 3)` свернуть по `axis=0` → останется `(3,)`.
`(4, 3)` свернуть по `axis=1` → останется `(4,)`.

In [ ]:
print("было:            ", table.shape)
print("sum(axis=0) ->   ", table.sum(axis=0).shape, table.sum(axis=0))
print("sum(axis=1) ->   ", table.sum(axis=1).shape, table.sum(axis=1))
print("sum() без axis ->", np.shape(table.sum()), table.sum())

### Почему все путаются

Фразу «`axis=0` — это по строкам» можно понять двумя способами, и
интуиция выбирает неверный. `axis=0` означает «**идти вдоль** оси 0,
то есть сверху вниз по строкам, схлопывая их» — и в результате остаются
**столбцы**, а не строки.

На осмысленных данных путаницы не возникает. Оценки: строка — студент,
столбец — семинар.

In [ ]:
grades = np.array([
    [8.0, 10.0, 6.0],     # Алиса
    [7.0,  7.0, 9.0],     # Борис
    [10.0, 9.0, 10.0],    # Виктор
])
students = ["Алиса", "Борис", "Виктор"]
seminars = ["сем. 1", "сем. 2", "сем. 3"]

print("shape:", grades.shape, "= (студентов, семинаров)\n")

by_seminar = grades.mean(axis=0)     # схлопнули студентов
print("axis=0 — средняя ПО СЕМИНАРУ (студенты исчезли):")
for name, value in zip(seminars, by_seminar):
    print(f"   {name}: {value:.2f}")

by_student = grades.mean(axis=1)     # схлопнули семинары
print("\naxis=1 — средняя ПО СТУДЕНТУ (семинары исчезли):")
for name, value in zip(students, by_student):
    print(f"   {name}: {value:.2f}")

Проверка себя: что должно остаться? Если ответ «по одному числу на
студента» — сворачиваем семинары, а семинары это ось 1.

### `keepdims` — когда ось нужна, но размером 1

Нужно, когда результат участвует в арифметике с исходным массивом:
broadcasting требует совпадения по осям.

In [ ]:
print("без keepdims:", grades.mean(axis=1).shape)
print("с keepdims:  ", grades.mean(axis=1, keepdims=True).shape)

centred = grades - grades.mean(axis=1, keepdims=True)
print("\nоценки минус собственная средняя каждого студента:")
print(centred.round(2))

Без `keepdims` тут была бы ошибка формы: `(3, 3)` и `(3,)` не
складываются так, как нужно.

### Три оси и больше

Правило то же. В `minipipe/distance.py` получается куб
`(точек, центров, признаков)`, и `sum(axis=2)` схлопывает признаки:

In [ ]:
cube = np.arange(24, dtype=float).reshape(2, 3, 4)
print("shape:", cube.shape)

for axis in (0, 1, 2):
    print(f"  sum(axis={axis}) -> {cube.sum(axis=axis).shape}")

`axis` есть не только у `sum`. Он одинаково работает у `min`, `max`,
`mean`, `std`, `argmin`, `argmax`, `cumsum` — и означает ровно то же.

In [ ]:
print("argmin(axis=1) — индекс худшего семинара у каждого студента:", grades.argmin(axis=1))
print("argmax(axis=0) — кто лучший на каждом семинаре:             ", grades.argmax(axis=0))

### `reshape`

Меняет форму, не трогая данные. `-1` означает «посчитай сам».

In [ ]:
column = np.arange(6, dtype=float).reshape(-1, 1)
print("столбик:", column.shape)
print("обратно в одномерный:", column.ravel())

## 3. Векторизация вместо цикла

Главное правило: **цикл по элементам — почти всегда ошибка.**

Мерить руками через `time.perf_counter()` неточно: один запуск ловит
случайный шум, а первый вызов вдобавок платит за прогрев. Для замеров
есть `timeit` — он гоняет код много раз и берёт лучшее время.

In [ ]:
import math
import timeit

data = np.random.default_rng(0).random(100_000)
as_list = data.tolist()

RUNS = 20


def compare(name: str, python_code: str, numpy_code: str) -> None:
    scope = {"data": data, "as_list": as_list, "np": np, "math": math}
    python_time = min(timeit.repeat(python_code, globals=scope, number=RUNS, repeat=5)) / RUNS
    numpy_time = min(timeit.repeat(numpy_code, globals=scope, number=RUNS, repeat=5)) / RUNS
    print(f"{name:20} python {python_time * 1000:7.2f} мс | "
          f"numpy {numpy_time * 1000:6.3f} мс | быстрее в {python_time / numpy_time:4.0f} раз")


compare("x * 2 + 1", "[x * 2 + 1 for x in as_list]", "data * 2 + 1")
compare("sqrt каждого", "[math.sqrt(x) for x in as_list]", "np.sqrt(data)")
compare("сумма квадратов", "sum(x * x for x in as_list)", "(data ** 2).sum()")
compare("сколько > 0.5", "sum(1 for x in as_list if x > 0.5)", "(data > 0.5).sum()")

Арифметика применяется поэлементно ко всему массиву сразу. Сравнения —
тоже, и дают массив из `True`/`False`:

In [ ]:
values = np.array([1, 5, 3, 8, 2])
print(values > 3)
print((values > 3).sum(), "элементов больше трёх")

## 4. Broadcasting

Операции над массивами разной формы. numpy достраивает недостающее сам —
это не удобство, а основной способ писать без циклов.

### Правило

1. Формы выравниваются **справа**. Слева недостающие оси считаются `1`.
2. По каждой оси размеры должны либо **совпадать**, либо один из них
   должен быть **`1`** — тогда он растягивается.
3. Иначе ошибка.

| A | B | выравнивание | результат |
|---|---|---|---|
| `(4, 3)` | `(3,)` | `(4,3)` и `(1,3)` | `(4, 3)` |
| `(4, 3)` | `(4, 1)` | — | `(4, 3)` |
| `(3, 1)` | `(1, 4)` | — | `(3, 4)` |
| `(2, 3, 4)` | `(3, 1)` | `(2,3,4)` и `(1,3,1)` | `(2, 3, 4)` |
| `(4, 3)` | `(4,)` | `(4,3)` и `(1,4)` | **ошибка** |

Последняя строка — самая частая ошибка: хотели вычесть по одному числу
на строку, а numpy выравнивает справа и пытается приложить к столбцам.

In [ ]:
table = np.arange(12, dtype=float).reshape(4, 3)

print("матрица:")
print(table)

print("\nмассив + число:")
print(table + 100)

print("\nмассив (4,3) + строка (3,) — по столбцам:")
print(table + np.array([0, 10, 20]))

print("\nмассив (4,3) + столбец (4,1) — по строкам:")
print(table + np.array([[0], [100], [200], [300]]))

### Что не сработает

In [ ]:
try:
    table + np.array([0, 10, 20, 30])          # (4,3) и (4,)
except ValueError as exc:
    print("ValueError:", exc)

print("\nлечится превращением в столбец:")
print(table + np.array([0, 10, 20, 30]).reshape(4, 1))

Ошибку читать так: `(4,3)` и `(4,)` выровнялись как `(4,3)` и `(1,4)`,
справа `3` против `4` — не совпало и ни одно не равно единице.

### `None` вставляет ось

`None` (он же `np.newaxis`) в индексе добавляет ось размера 1. Ничего не
копируется — это view.

In [ ]:
values = np.arange(4, dtype=float)

print("исходный:      ", values.shape)
print("values[:, None]:", values[:, None].shape, "— столбец")
print("values[None, :]:", values[None, :].shape, "— строка")

Столбец плюс строка дают таблицу — каждый с каждым:

In [ ]:
rows = np.array([1.0, 2.0, 3.0])
cols = np.array([10.0, 20.0, 30.0, 40.0])

print((rows[:, None] * cols[None, :]).shape)
print(rows[:, None] * cols[None, :])

Вот это «каждый с каждым» и есть главный приём. Дальше — он же, только
на трёх осях.

### Практика: вычесть среднее по столбцу

In [ ]:
grades = np.array([
    [8.0, 10.0, 6.0],
    [7.0,  7.0, 9.0],
    [10.0, 9.0, 10.0],
])

by_seminar = grades.mean(axis=0)        # (3,) — среднее каждого семинара
print("средние по семинарам:", by_seminar)
print("\nотклонение от средней по семинару:")
print((grades - by_seminar).round(2))   # (3,3) - (3,) -> ок, выравнивание справа

А чтобы вычесть среднее **по студенту**, нужен `keepdims` из раздела 2.
И вот тут главная ловушка broadcasting: без него **ошибки не будет** —
формы выровняются, и результат окажется тихо неверным.

In [ ]:
by_student = grades.mean(axis=1)        # (3,) — среднее каждого студента

# Ошибки не будет: (3,3) и (3,) выровняются справа, и вычтется НЕ ТО —
# из первого семинара средняя первого студента, и так далее.
print("без keepdims — тихо неверно:")
print((grades - by_student).round(2))

print("\nс keepdims — правильно, по строкам:")
print((grades - grades.mean(axis=1, keepdims=True)).round(2))

### Разбор конструкции из `minipipe`

Расстояния от каждой точки до каждого центра. Ровно тот же приём
«каждый с каждым», но объекты — не числа, а векторы признаков.

In [ ]:
points = np.array([[0.0, 0.0], [1.0, 1.0], [5.0, 5.0]])    # (3, 2)
centers = np.array([[0.0, 0.0], [5.0, 5.0]])               # (2, 2)

print("points[:, None, :] ->", points[:, None, :].shape)   # (3, 1, 2)
print("centers[None, :, :] ->", centers[None, :, :].shape) # (1, 2, 2)

difference = points[:, None, :] - centers[None, :, :]      # (3, 2, 2)
print("разность            ->", difference.shape, "= (точек, центров, признаков)")

squared = (difference ** 2).sum(axis=2)                    # (3, 2)
print("после sum(axis=2)   ->", squared.shape, "= (точек, центров)")
print(squared)

По шагам: `(3,1,2)` и `(1,2,2)` выравниваются в `(3,2,2)` — куб
«точка × центр × признак». Вычитание поэлементное, `sum(axis=2)`
схлопывает признаки, остаётся матрица «точка × центр».

Ни одного цикла на Python — вместо двух вложенных.

### Чем платим

Промежуточный куб `(точек, центров, признаков)` создаётся **целиком, в
памяти**. Это и есть цена отказа от цикла.

In [ ]:
for n_points, n_centers, n_features in [(1_000, 8, 10), (100_000, 100, 50)]:
    size = n_points * n_centers * n_features * 8      # float64 — 8 байт
    print(f"{n_points:>7} точек x {n_centers:>3} центров x {n_features:>2} признаков "
          f"= {size / 1024**2:8.1f} МБ")

На учебных данных незаметно, на реальных — почти четыре гигабайта за
одну строчку кода. Как с этим жить, разберём на семинаре про память.

## 5. Индексация

Три способа, и все три встречаются в `minipipe`.

In [ ]:
values = np.arange(10, dtype=float)

print("срез:           ", values[2:5])
print("с конца:        ", values[-3:])
print("шаг:            ", values[::2])

**Список индексов** — берём элементы в нужном порядке. Так работает
перемешивание в `prep.shuffled` и деление выборки в `split`:

In [ ]:
order = np.array([3, 1, 4, 0])
print("values[order]:", values[order])

**Булева маска** — берём то, что подошло под условие. Так `kmeans`
собирает точки своего кластера:

In [ ]:
labels = np.array([0, 1, 0, 1, 0])
points = np.arange(10, dtype=float).reshape(5, 2)

print(points)
print("\nточки кластера 0:")
print(points[labels == 0])
print("\nих среднее:", points[labels == 0].mean(axis=0))

Для таблиц индекс двумерный: `table[строки, столбцы]`.

In [ ]:
table = np.arange(12, dtype=float).reshape(4, 3)
print("одна ячейка:   ", table[1, 2])
print("вся строка:    ", table[1])
print("весь столбец:  ", table[:, 2])
print("подтаблица:\n", table[1:3, 0:2])

## 6. View против copy

Срез **не копирует** данные — это окно в тот же массив. Пишешь в срез —
меняется оригинал. Самая частая ловушка numpy.

In [ ]:
original = np.arange(6)
window = original[1:4]
window[0] = 999

print("original:", original)
print("это view?", window.base is original)

In [ ]:
original = np.arange(6)
copied = original[1:4].copy()
copied[0] = 999

print("original:", original, "— цел")

Индексация списком и маской, наоборот, **всегда копирует**:

In [ ]:
original = np.arange(6)
taken = original[[1, 2, 3]]
taken[0] = 999
print("original:", original, "— цел, fancy indexing копирует")

## 7. Полезные функции

Те, что встретятся в коде курса.

In [ ]:
distances = np.array([5.0, 1.0, 3.0, 2.0])

print("argmin — индекс минимума:", distances.argmin())
print("argsort — порядок сортировки:", np.argsort(distances))
print("три ближайших:", np.argsort(distances)[:3])
print("три дальних:  ", np.argsort(distances)[-3:])

In [ ]:
labels = np.array([0, 1, 1, 0, 1])

print("bincount — сколько каждого:", np.bincount(labels))
print("самая частая метка:", np.bincount(labels).argmax())

In [ ]:
a = np.array([0.1 + 0.2, 1.0])
b = np.array([0.3, 1.0])

print("a == b      :", a == b)          # поэлементно, и первый False
print("np.allclose :", np.allclose(a, b))

`==` на float не работает по той же причине, что и в обычном Python.
Для массивов есть `np.allclose`, для тестов — `pytest.approx`.

In [ ]:
values = np.array([1.5, 2.7, 3.2])

print("astype — смена типа:", values.astype(np.int64))
print("zeros_like — нули той же формы:", np.zeros_like(values))
print("\nnan заразен:", np.array([1.0, np.nan, 3.0]).sum())
print("isnan:", np.isnan(np.array([1.0, np.nan, 3.0])))

## 8. Случайность

Современный способ — генератор с явным seed. `np.random.seed()` —
устаревший, глобальный и поэтому опасный.

In [ ]:
generator = np.random.default_rng(42)
print(generator.random(3))

print("тот же seed — тот же результат:", np.random.default_rng(42).random(3))
print("другой seed:                   ", np.random.default_rng(7).random(3))

In [ ]:
generator = np.random.default_rng(0)

print("permutation — случайный порядок индексов:", generator.permutation(6))
print("choice — выбрать k без повторов:", generator.choice(10, 3, replace=False))
print("normal — нормальное распределение:", generator.normal(0, 1, 3).round(3))

## 9. Где это в `minipipe`

| Конструкция | Где |
|---|---|
| `points[:, None, :] - centers[None, :, :]` | `distance.py` |
| `.sum(axis=2)`, `.argmin(axis=1)` | `distance.py`, `kmeans.py` |
| `values[order]` | `prep.shuffled`, `split.train_test_split` |
| `points[labels == index]` | `kmeans._recompute` |
| `.mean(axis=0)` | `kmeans._recompute` |
| `np.argsort(...)[:k]` | `knn.predict` |
| `np.bincount(...).argmax()` | `knn.predict` |
| `np.allclose` | `kmeans.fit` — проверка сходимости |
| `np.zeros_like`, `np.empty` | `prep.py`, `knn.py` |
| `default_rng(seed)`, `.permutation`, `.choice` | `prep.py`, `split.py`, `kmeans.py` |
| `.astype(np.int64)` | везде, где нужны метки |

Открой [`minipipe/distance.py`](../minipipe/distance.py) — теперь он
читается целиком.